<a href="https://colab.research.google.com/github/kimve1969/PINN/blob/main/pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Примеры работы с тензорами

In [ ]:
import torch

A = torch.tensor([[1.0, 2],[3,4]])
b = torch.tensor([[5.0],[6]])
x = torch.linalg.solve(A,b)
print(x)

In [ ]:
A

In [ ]:
A.T

In [ ]:
torch.matmul(A, A.T)

In [ ]:
A[0,:]

In [ ]:
A[:,0]

In [ ]:
A[A>2]

# Автоматическое дифференцирование

$
z(x,y)=x*y+x^3 \\
\frac{\partial z(x,y)}{\partial x} = y + 3*x^2 \\
\frac{\partial z(x,y)}{\partial y} = x
$

## Дифф. через backward() - это функция обратного распространения ошибки
позволяет дифф. только скалярные функции

In [ ]:
import torch

# То же самое, что и выше только через функцию обратное распространение оштибки
x = torch.tensor([2.], requires_grad=True)
y = torch.tensor([3.], requires_grad=True)

z = x * y + x**3
# Вычисляет все частные производные, значения ЧП нах-ся в атрибутах переменных variable.grad
z.backward()
z

In [ ]:
z = x * y + x**3
z

In [ ]:
# grad_outputs=torch.ones_like(x) - результат дифф. будет матрица dz_dx
# create_graph=True, этот параметр необходим для дифф. второго порядка d2z_dx2
dz_dx = torch.autograd.grad(z, x, grad_outputs=torch.ones_like(x), create_graph=True, retain_graph=True)
dz_dx

In [ ]:
d2z_dx2 = torch.autograd.grad(dz_dx, x, grad_outputs=torch.ones_like(x), retain_graph=True)
d2z_dx2

## Вычисляем Якобиан "вручную" (autograd.grad() в цикле)

In [ ]:
import torch

x = torch.tensor([1., 2.], requires_grad=True)
# z - vector-function (is not scalar)
z = torch.stack([ x[0] * x[1] + x[0]**3, x[1]**3, torch.sin( torch.pi * x[0])])
z

In [ ]:
jacobian = torch.zeros([z.shape[0], x.shape[0]])
jacobian

In [ ]:
# jacobian
# dz1/dx1, dz1/dx2
# dz2/dx1, dz2/dx2
# dz3/dx1, dz3/dx2

for i in range( z.shape[0] ):
    jacobian[i] = torch.autograd.grad( outputs=z[i], inputs=x, grad_outputs=torch.ones_like(z[i]), retain_graph=True, create_graph=False)[0]
    #print(jacobian[i])
    #print(type(jac_i))

print(jacobian)

## Вычисляем Якобиан через jacobian()

In [ ]:
import torch
import torch.autograd.functional

def z(x):
    return torch.stack([ x[0] * x[1] + x[0]**3, x[1]**3, torch.sin( torch.pi * x[0])])

print(type(z))

x=torch.tensor([1.,2.], requires_grad=True)

z=jacobian(func=z, inputs=x, create_graph=True)
print(z)

# Работа с GPU и CUDA

In [ ]:
import torch

# Проверка доступности CUDA
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")

In [ ]:
torch.cuda.memory_allocated()

In [ ]:
import torchvision
# import torchaudio

# Матричное умножение на CPU & GPU (очень ресурсоемкие операции !!!)

In [ ]:
# Тайминг на Yandex DataSphere
# Конфигурация              Операция         Тайминг
# CPU 8 core, GPU V100      cpu A*B          40 s
#                           A cpu->gpu       1 s
#                           gpu A*B          60 ms
#                           C gpu->cpu       2 min 43 s
# CPU 28 core, GPU A100     cpu A*B          37 s
#                           A cpu->gpu       0.5 s
#                           gpu A*B          200 ms
#                           C gpu->cpu       1 min 20 s
# Server DGX-1 SNG, V100    cpu A*B          13,5 s
#                           A cpu->gpu       1 s
#                           gpu A*B          70 ms
#                           C gpu->cpu       3 min 8 s

import torch

# matrix multiple on CPU
MAXSIZE = 24000
%time A = torch.rand(MAXSIZE,MAXSIZE)
%time B = torch.rand(MAXSIZE,MAXSIZE)

%time C = torch.matmul(A,B)

In [ ]:
# matrix multiple on GPU
%time A_cuda = A.to("cuda")
%time B_cuda = B.to("cuda")
for i in range(100):
    %time C_cuda = torch.matmul(A_cuda,B_cuda)

In [ ]:
%time C_cuda.device

In [ ]:
# переносим обратно с GPU на CPU
%time C_cpu = C_cuda.to("cpu")
%time C_cpu.device

In [ ]:
# Смотрим разницу вычислений на GPU на CPU, должно быть небольшое значение
%time C_res = C - C_cpu
%time torch.norm(C_res)

In [ ]:
C_res[0:10, 0:10]

In [ ]:
# удаляем данные
%time del A_cuda, B_cuda, C_cuda